In [2]:
import pandas as pd
import requests
from io import StringIO

# SHFE prices

In [16]:
response = requests.get (f"https://www.shfe.com.cn/data/tradedata/future/dailydata/kx20260522.dat")

In [17]:
data = response.json()
df = pd.DataFrame(data['o_curinstrument'])

df["product_group_index"] = pd.factorize(df["PRODUCTGROUPID"])[0] # Create a new column with the group index

df = df.sort_values(["product_group_index", "DELIVERYMONTH"], ascending=[True, True], kind='stable') # Sort by group index and delivery month
df.drop(columns=["product_group_index"], inplace=True) # Remove the temporary group index column
df.reset_index(drop=True, inplace=True) # Reset the index after sorting

df.insert(0,"M",df.groupby("PRODUCTGROUPID").cumcount() + 1) # Add a new column "M" with the cumulative count of rows within each group, starting from 1
sub_totals_index = df.groupby("PRODUCTGROUPID").tail(1).index # Get the index of the last row of each group which turns to be the subtotal row
df.loc[sub_totals_index, "M"] = 0 # Set the "M" value to 0 for the subtotal rows

In [18]:
df.to_csv("shfe_prices.csv", index=False, encoding="utf-8-sig")

# SHFE stocks

In [12]:
response = requests.get(f"https://www.shfe.cn/data/tradedata/future/stockdata/weeklystock_20260522/EN/all.html")

In [15]:
tables = pd.read_html(StringIO(response.text)) # Delivers a list of dataframes, one for each table found in the HTML
tables[1]
for table in tables:
    print(table.iloc[0,0]) # Print the first cell of each table to identify which one contains the desired data

Issue No.(19),2026,(Total Issues1386) DATE:2026-05-22
COPPER
ALUMINUM
ZINC
LEAD
NICKEL
TIN
Aluminlum Oxide(warehourse)
Aluminlum Oxide(factory warehouse)
Casting Aluminium Alloy
NATURAL RUBBER
Butadiene Rubber(warehourse)
Butadiene Rubber(factory warehouse)
Pulp(warehourse)
Pulp(factory warehouse)
OFFSET PAPER(warehourse)
OFFSET PAPER(factory warehouse)
FUEL OIL
BITUMEN(warehourse)
BITUMEN(factory warehouse)
GOLD
SILVER
Rebar(warehourse)
Rebar(factory warehouse)
WIRE ROD
HOT ROLLED COILS(warehourse)
HOT ROLLED COILS(factory warehouse)
Stainless Steel(warehourse)
Stainless Steel(factory warehouse)
Note: 1. Theoretical available storage capacity is for the entire depot, but not by crude. 2. The Basra light crude oil(levelⅠ) is the Basra light that has an API greater than or equal to 28 and sulfur content lower than or equal to 3.5%. The Basra light crude oil(levelⅡ) is the Basra light that has an API greater than or equal to 28 and sulfur content between 3.5% and 4.0%(4.0% included), or 

In [23]:
for table in tables:
    df = pd.concat([df, table], ignore_index=True) # Concatenate all tables into a single dataframe

In [20]:
# SHFE old version
response = requests.get(f"https://www.shfe.cn/data/tradedata/future/weeklydata/20250905weeklystock.dat")

In [21]:
data = response.json()
df = pd.DataFrame(data['o_cursor'])

In [22]:
df.columns

Index(['WHSTOCKS', 'WGHTUNIT', 'WHABBRNAME', 'SPOTWGHTS', 'ROWSTATUS',
       'PREWHSTOCKS', 'WHSTOCKCHANGE', 'REGNAME', 'PREWRTWGHTS', 'SPOTCHANGE',
       'VARNAME', 'WHROWS', 'WHTYPE', 'VARID', 'WRTWGHTS', 'WRTWEEKRPTTYPE',
       'WRTCHANGE', 'PRESPOTWGHTS'],
      dtype='str')

In [16]:
print(df["VARID"].unique())
print(df["VARNAME"].unique())

<StringArray>
['cu', 'al', 'zn', 'pb', 'ni', 'sn', 'ao', 'ru', 'br', 'sp', 'bu', 'au', 'ag',
 'rb', 'wr', 'hc', 'ss']
Length: 17, dtype: str
<StringArray>
[                                    '铜$$COPPER',
                                  '铝$$ALUMINIUM',
                                       '锌$$ZINC',
                                       '铅$$LEAD',
                                     '镍$$NICKEL',
                                        '锡$$TIN',
            '氧化铝(仓库)$$Aluminium Oxide Warehouse',
    '氧化铝(厂库)$$Aluminium Oxide Factory Warehouse',
                          '天然橡胶$$NATURAL RUBBER',
         '丁二烯橡胶(仓库)$$Butadiene Rubber warehouse',
 '丁二烯橡胶(厂库)$$Butadiene Rubber factory warehouse',
                        '纸浆(仓库)$$Pulp Warehouse',
                '纸浆(厂库)$$Pulp Factory Warehouse',
                   '石油沥青(仓库)$$BITUMEN Warehouse',
           '石油沥青(厂库)$$BITUMEN Factory Warehouse',
                                      '黄金$$GOLD',
                                    '白银$$SILV